# Chapter 08: Closed-Loop Control & Vehicle Kinematics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvknyn/self-driving-ai-course/blob/main/notebooks/08_closed_loop_control_stanley.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-181717.svg)](https://github.com/vvknyn/self-driving-ai-course)

> **The Big Question**: *Why do simple steering controllers enter divergent death wobbles at 70 mph, and how does Stanley control prevent it?*

---

## 1. 🚨 The Real-World Dilemma
Vehicle heading rate $\dot{\psi} = \frac{v}{L} \tan(\delta)$ scales with speed. A steering angle that works at 10 mph rolls the car at 70 mph! The Stanley Controller dampens cross-track steering by velocity in the denominator: $\arctan(\frac{ke}{v + k_{\text{soft}}})$, guaranteeing exponential convergence without oscillation.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

class KinematicBicycle:
    def __init__(self, x=0.0, y=0.5, yaw=0.0, v=25.0, L=2.8):
        self.x = x
        self.y = y
        self.yaw = yaw
        self.v = v
        self.L = L

    def step(self, delta, dt=0.02):
        self.x += self.v * np.cos(self.yaw) * dt
        self.y += self.v * np.sin(self.yaw) * dt
        self.yaw += (self.v / self.L) * np.tan(delta) * dt

def stanley_control(vehicle, path_y=0.0, k=0.8, k_soft=1.0):
    fx = vehicle.x + vehicle.L * np.cos(vehicle.yaw)
    fy = vehicle.y + vehicle.L * np.sin(vehicle.yaw)
    cross_track_error = path_y - fy
    heading_error = -vehicle.yaw
    delta = heading_error + np.arctan2(k * cross_track_error, vehicle.v + k_soft)
    return np.clip(delta, -np.deg2rad(30), np.deg2rad(30))

car = KinematicBicycle()
history_y = []
for _ in range(200):
    delta = stanley_control(car)
    car.step(delta)
    history_y.append(car.y)

plt.figure(figsize=(8, 3))
plt.plot(history_y, color="#3fb950", lw=2, label="Vehicle Lateral Position (Stanley)")
plt.axhline(0.0, color="k", linestyle="--", label="Target Path Centerline")
plt.title("Stanley Controller Exponential Cross-Track Convergence at 55 mph")
plt.xlabel("Simulation Steps (dt = 20ms)")
plt.ylabel("Lateral Error (m)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 2. 🩺 Andrew Ng's Diagnostic Field Guide

| Observed Symptom | Underlying Mathematical Mechanism | Verification Test | Production Fix |
| :--- | :--- | :--- | :--- |
| **Steady-state cross-track offset on banked highways** | Gravitational lateral force $g \sin(\phi)$ not canceled by pure P-term. | Measure mean error over constant bank. | Add integral anti-windup term ($K_i \int e \, dt$). |
| **Steering shudder at near-zero speeds ($v < 0.2\text{ m/s}$)** | Singularity at zero velocity when $k_{\text{soft}} = 0$. | Check steering commands when coming to a complete stop. | Set $k_{\text{soft}} \ge 1.0\text{ m/s}$. |